# Tome 2, Chapter 2: Morphisms and Quotients (Python companion)

Exercises first, solutions at the end. We find normal subgroups by brute force, build quotient groups and
semidirect products as multiplication tables, tell the five groups of order $8$ apart, and use the Chinese remainder
isomorphism the way every RSA implementation does: **decrypting modulo $p$ and $q$ separately is about three times faster**.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
import time
from collections import Counter
import sympy

## Exercises

**P2.1 (★) Normal subgroups.** Write a function listing all subgroups of a small permutation group (closure of pairs of elements is enough for the groups below) and marking the normal ones. Apply it to $\mathfrak S_3$, $D_4$, $A_4$ and $\mathfrak S_4$. *Guess first: does $A_4$ have a normal subgroup of order $4$?*

In [ ]:
# Your code here


**P2.2 (★) A quotient by hand.** In $D_4$, with $N=\langle r^2\rangle$, list the cosets and the multiplication table of $D_4/N$ (Theorem 2.6). Which group of order $4$ is it (Exercise 1.3)? Then try the same rule with the non-normal $H=\langle s\rangle$ and show that it is not well defined.

In [ ]:
# Your code here


**P2.3 (★★) The five groups of order 8.** Build $\Z/8\Z$, $\Z/4\Z\times\Z/2\Z$, $(\Z/2\Z)^3$, $D_4$ and $Q_8$ in Python, and for each print the number of elements of each order and whether it is abelian (Exercise 2.8).

In [ ]:
# Your code here


**P2.4 (★★) Semidirect products.** For $u\in(\Z/n\Z)^*$ with $u^m\equiv1\pmod n$, the rule $\varphi_h(k)=u^hk$ defines $\Z/n\Z\rtimes\Z/m\Z$. Implement its product, check associativity by brute force, and find for which $u$ the group $\Z/7\Z\rtimes\Z/3\Z$ is not abelian. Compare its order statistics with $\Z/21\Z$.

In [ ]:
# Your code here


**P2.5 (★★★) CS application: RSA with the Chinese remainder theorem.** RSA decrypts with $m=c^d\bmod N$, $N=pq$. By Theorem 2.11, $\Z/N\Z\cong\Z/p\Z\times\Z/q\Z$, so one can compute $c^{d\bmod(p-1)}\bmod p$ and $c^{d\bmod(q-1)}\bmod q$ and recombine them (Garner's formula). Implement both with $1024$-bit primes and time them.

*How to think about it:* modular exponentiation with $k$-bit numbers costs about $k^3$ bit operations (exponent of $k$ bits, each multiplication about $k^2$). Two half-size exponentiations cost $2\cdot(k/2)^3=k^3/4$.

In [ ]:
# Your code here


---
## Solutions

### P2.1

In [ ]:
def compose(p, q):
    return tuple(p[i] for i in q)

def inverse(p):
    inv = [0] * len(p)
    for i, x in enumerate(p):
        inv[x] = i
    return tuple(inv)

def closure(gens, n):
    e = tuple(range(n))
    H, frontier = {e}, [e]
    while frontier:
        new = []
        for h in frontier:
            for g in gens:
                x = compose(g, h)
                if x not in H:
                    H.add(x); new.append(x)
        frontier = new
    return frozenset(H)

def subgroups(G, n):
    return {closure([a, b], n) for a in G for b in G}

def is_normal(H, G):
    return all(compose(compose(g, h), inverse(g)) in H for g in G for h in H)

def sign(p):
    return (-1) ** sum(1 for i in range(len(p)) for j in range(i + 1, len(p)) if p[i] > p[j])

def dihedral(n):
    r = tuple((i + 1) % n for i in range(n)); s = tuple((-i) % n for i in range(n))
    return sorted(closure([r, s], n))

S3 = list(itertools.permutations(range(3)))
S4 = list(itertools.permutations(range(4)))
A4 = [p for p in S4 if sign(p) == 1]
for name, G, n in [("S_3", S3, 3), ("D_4", dihedral(4), 4), ("A_4", A4, 4), ("S_4", S4, 4)]:
    subs = subgroups(G, n)
    normal = sorted(len(H) for H in subs if is_normal(H, G))
    print(f"{name}: {len(subs)} subgroups, orders of the normal ones: {normal}")

$A_4$ has a normal subgroup of order $4$: the Klein group $V=\{e,(1\,2)(3\,4),(1\,3)(2\,4),(1\,4)(2\,3)\}$, normal even in $\mathfrak S_4$. So $A_4/V\cong\Z/3\Z$, and $\mathfrak S_4/V\cong\mathfrak S_3$: this is the map behind the cubic resolvent of a quartic, which we meet again in Chapter 10. $\mathfrak S_4$ has exactly $4$ normal subgroups: $\{e\},V,A_4,\mathfrak S_4$.

### P2.2

In [ ]:
D4 = dihedral(4)
r = (1, 2, 3, 0); s = (0, 3, 2, 1); e = (0, 1, 2, 3)
N = closure([compose(r, r)], 4)
cosets = []
for g in D4:
    C = frozenset(compose(g, x) for x in N)
    if C not in cosets:
        cosets.append(C)
print(len(cosets), "cosets of <r^2>")
def coset_of(g, cos):
    return next(i for i, C in enumerate(cos) if g in C)
for i, A in enumerate(cosets):
    row = []
    for j, B in enumerate(cosets):
        products = {coset_of(compose(a, b), cosets) for a in A for b in B}
        assert len(products) == 1          # well defined
        row.append(products.pop())
    print(row)

H = closure([s], 4)
cosH = []
for g in D4:
    C = frozenset(compose(g, x) for x in H)
    if C not in cosH:
        cosH.append(C)
bad = [(i, j) for i, A in enumerate(cosH) for j, B in enumerate(cosH)
       if len({coset_of(compose(a, b), cosH) for a in A for b in B}) > 1]
print("with H = <s>: pairs of cosets whose product is not well defined:", len(bad), "of", len(cosH) ** 2)

Every element of $D_4/\langle r^2\rangle$ squares to the identity (the diagonal of the table is $0$), so it is the Klein four-group. With $H=\langle s\rangle$, which is not normal, half of the products of cosets depend on the representatives chosen: exactly what Theorem 2.6 predicts.

### P2.3

In [ ]:
def order_stats(elements, mul, e):
    def order(g):
        k, x = 1, g
        while x != e:
            x = mul(x, g); k += 1
        return k
    abelian = all(mul(a, b) == mul(b, a) for a in elements for b in elements)
    return dict(sorted(Counter(order(g) for g in elements).items())), abelian

Z8 = (list(range(8)), lambda a, b: (a + b) % 8, 0)
Z4Z2 = (list(itertools.product(range(4), range(2))), lambda a, b: ((a[0] + b[0]) % 4, (a[1] + b[1]) % 2), (0, 0))
Z2cube = (list(itertools.product(range(2), repeat=3)), lambda a, b: tuple((x + y) % 2 for x, y in zip(a, b)), (0, 0, 0))
D4g = (D4, compose, e)
# Q8 as pairs (sign, unit) with units 1, i, j, k
table = {('1', u): (1, u) for u in '1ijk'}
table.update({(u, '1'): (1, u) for u in '1ijk'})
table.update({('i', 'i'): (-1, '1'), ('j', 'j'): (-1, '1'), ('k', 'k'): (-1, '1'),
              ('i', 'j'): (1, 'k'), ('j', 'k'): (1, 'i'), ('k', 'i'): (1, 'j'),
              ('j', 'i'): (-1, 'k'), ('k', 'j'): (-1, 'i'), ('i', 'k'): (-1, 'j')})
def qmul(a, b):
    sgn, u = table[(a[1], b[1])]
    return (a[0] * b[0] * sgn, u)
Q8 = ([(sg, u) for sg in (1, -1) for u in '1ijk'], qmul, (1, '1'))
for name, (G, mul, e0) in [("Z/8", Z8), ("Z/4 x Z/2", Z4Z2), ("(Z/2)^3", Z2cube), ("D_4", D4g), ("Q_8", Q8)]:
    stats, ab = order_stats(G, mul, e0)
    print(f"{name:10s} orders {stats}  abelian: {ab}")

The order statistics already separate the five groups: only $D_4$ has five elements of order $2$, only $Q_8$ has six of order $4$ and a single one of order $2$. (Order statistics do not always separate groups: the smallest counterexample has order $16$.)

### P2.4

In [ ]:
def semidirect(n, m, u):
    elems = [(k, h) for k in range(n) for h in range(m)]
    mul = lambda a, b: ((a[0] + pow(u, a[1], n) * b[0]) % n, (a[1] + b[1]) % m)
    return elems, mul, (0, 0)

for u in range(1, 7):
    if pow(u, 3, 7) != 1:
        continue
    G, mul, e0 = semidirect(7, 3, u)
    assoc = all(mul(mul(a, b), c) == mul(a, mul(b, c)) for a in G for b in G for c in G)
    stats, ab = order_stats(G, mul, e0)
    print(f"u = {u}: associative {assoc}, abelian {ab}, orders {stats}")
print("Z/21:", order_stats(list(range(21)), lambda a, b: (a + b) % 21, 0)[0])

The cube roots of $1$ modulo $7$ are $1,2,4$. For $u=1$ we get $\Z/7\Z\times\Z/3\Z\cong\Z/21\Z$; for $u=2$ and $u=4$, a non-abelian group of order $21$, with $14$ elements of order $3$ and $6$ of order $7$ and none of order $21$. The two choices give isomorphic groups (replace the generator $h$ by $2h$): it is the unique non-abelian group of order $21$, as Chapter 5 will show.

### P2.5

In [ ]:
random.seed(1)
p = sympy.randprime(2**1023, 2**1024)
q = sympy.randprime(2**1023, 2**1024)
N, phi = p * q, (p - 1) * (q - 1)
e_pub = 65537
d = pow(e_pub, -1, phi)
m = random.randrange(N)
c = pow(m, e_pub, N)

def decrypt_plain(c):
    return pow(c, d, N)

dp, dq, q_inv = d % (p - 1), d % (q - 1), pow(q, -1, p)
def decrypt_crt(c):
    mp, mq = pow(c, dp, p), pow(c, dq, q)
    h = q_inv * (mp - mq) % p          # Garner: the unique x mod N with x = mp mod p, x = mq mod q
    return mq + h * q

assert decrypt_plain(c) == decrypt_crt(c) == m
for f in (decrypt_plain, decrypt_crt):
    t = time.perf_counter()
    for _ in range(20):
        f(c)
    print(f"{f.__name__:14s}: {(time.perf_counter() - t) / 20 * 1000:.1f} ms per decryption")

The CRT version is about three to four times faster, close to the predicted factor $4$. Every serious RSA implementation (OpenSSL, the PKCS#1 private key format with its fields `exponent1`, `exponent2`, `coefficient`) stores exactly $d_p$, $d_q$ and $q^{-1}\bmod p$ for this reason. The isomorphism $\Z/N\Z\cong\Z/p\Z\times\Z/q\Z$ is also why RSA works at all: $m^{ed}\equiv m$ holds modulo $p$ and modulo $q$ separately (Fermat), hence modulo $N$.